# PhDAI 732 Group Project Part 1
**Group 5** | PhiUSIIL Phishing URL Dataset

This notebook is a thin driver. All logic lives in `src/`. If you find yourself
writing a function in a cell, it belongs in a module instead, or the next person
to run this notebook gets different numbers than you did.

In [1]:
# Setup. Run once per Colab session.
REPO = 'https://github.com/jecollier041/phdai_732_gp_phiusil.git'
DIR  = 'phdai_732_gp_phiusil'

import os, sys
if not os.path.exists(DIR):
    !git clone -q $REPO
%cd $DIR
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())

# Sanity check: this must print the repo root, and src must import.
from src import config as C
print('ROOT      :', C.ROOT)
print('RAW_CSV   :', C.RAW_CSV, '| present:', C.RAW_CSV.exists())
print('SEED      :', C.SEED, '| TEST_SIZE:', C.TEST_SIZE)


c:\Users\alber\Groupe_Project_1_phdai_732_gp_phiusil\phdai_732_gp_phiusil
ROOT      : C:\Users\alber\Groupe_Project_1_phdai_732_gp_phiusil\phdai_732_gp_phiusil
RAW_CSV   : C:\Users\alber\Groupe_Project_1_phdai_732_gp_phiusil\phdai_732_gp_phiusil\data\PhiUSIIL_Phishing_URL_Dataset.csv | present: False
SEED      : 42 | TEST_SIZE: 0.25


In [2]:
# Fetch the dataset into data/. Do not commit the CSV.
!pip install -q ucimlrepo
import pandas as pd
from src import config as C

if not C.RAW_CSV.exists():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=967)
    frame = ds.data.original.copy()
    frame = frame.loc[:, ~frame.columns.duplicated()]
    C.RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(C.RAW_CSV, index=False)
    # ucimlrepo splits identifier columns off into ds.data.ids on some
    # datasets. If FILENAME/URL/Domain/Title are missing, load_data will raise
    # a KeyError naming them, which is the signal that this cell needs fixing
    # rather than config.py.
print(C.RAW_CSV, C.RAW_CSV.exists())


C:\Users\alber\Groupe_Project_1_phdai_732_gp_phiusil\phdai_732_gp_phiusil\data\PhiUSIIL_Phishing_URL_Dataset.csv True


In [3]:
# Step 1 (Data steward): load, clean, document.
from src.data import load_data

df, log = load_data()
log

{'path': 'C:\\Users\\alber\\Groupe_Project_1_phdai_732_gp_phiusil\\phdai_732_gp_phiusil\\data\\PhiUSIIL_Phishing_URL_Dataset.csv',
 'rows_raw': 235795,
 'cols_raw': 56,
 'unaccounted_columns': ['TLD'],
 'duplicate_rows_removed': 0,
 'duplicate_urls_removed': 425,
 'urls_with_conflicting_labels': 0,
 'rows_clean': 235370,
 'columns_with_missing': {},
 'class_counts': {'1': 134850, '0': 100520},
 'n_unique_domains': 220086,
 'urlsimilarity_by_label': {0: {'mean': 49.651465963879204,
   'min': 0.155574352,
   'max': 100.0},
  1: {'mean': 100.0, 'min': 100.0, 'max': 100.0}},
 'label_value_for_legitimate': 1,
 'note': 'Confirm against the UCI documentation before citing in the report.',
 'fingerprint': '0f9dc76443ca80d8e2cd7ad7183795a7c4619c1b0b2d4a37a60635003a568a3b',
 'fingerprint_expected': '0f9dc76443ca80d8e2cd7ad7183795a7c4619c1b0b2d4a37a60635003a568a3b',
 'fingerprint_matches': True}

In [4]:
# Step 2 (Data steward): leakage screen. Run before anyone models.
from src.leakage import single_feature_accuracy, class_constancy

screen = single_feature_accuracy(df)
display(screen.head(12))
class_constancy(df).head(8)

,feature,single_feature_accuracy,flagged
0,URLSimilarityIndex,0.996661,True
1,NoOfExternalRef,0.961342,True
2,LineOfCode,0.955364,True
3,NoOfSelfRef,0.948515,False
4,NoOfImage,0.942639,False
5,NoOfJS,0.931669,False
6,NoOfCSS,0.905043,False
7,HasSocialNet,0.880125,False
8,HasCopyrightInfo,0.865688,False
9,HasDescription,0.830293,False


,feature,label,modal_value,modal_share
13,NoOfObfuscatedChar,1,0.0,1.0
15,ObfuscationRatio,1,0.0,1.0
11,HasObfuscation,1,0.0,1.0
5,IsDomainIP,1,0.0,1.0
29,NoOfAmpersandInURL,1,0.0,1.0
35,IsHTTPS,1,1.0,1.0
25,NoOfEqualsInURL,1,0.0,1.0
27,NoOfQMarkInURL,1,0.0,1.0


In [5]:
# Step 3: freeze the split. Do this once. Commit results/split_assignment.csv.
# Each strategy caches to its own file, so running the grouped split below
# does not silently hand back the stratified one.
from src.splits import make_split, get_xy

assignment = make_split(df, strategy='stratified')
print(assignment.value_counts().to_dict())

# Domain-grouped comparison. Separate cache, separate numbers.
assignment_grouped = make_split(df, strategy='grouped')
print(assignment_grouped.value_counts().to_dict())


{'train': 176527, 'test': 58843}
{'train': 176906, 'test': 58464}


In [6]:
# Step 4 (Modeling lead): all models across all three feature sets.
# Runtime note: random_forest is ~100 s per feature set on a fast multicore
# box and several times that on a free Colab CPU runtime. Budget ~45 min for
# this cell, or pass cv=False to evaluate() while iterating.
from src.models import build_models, evaluate
import pandas as pd

records = []
for fs in ['full', 'no_derived', 'url_only']:
    Xtr, Xte, ytr, yte = get_xy(df, assignment, feature_set=fs)
    for name, model in build_models().items():
        records.append(evaluate(model, name, fs, Xtr, Xte, ytr, yte))

results = pd.DataFrame(records)
results[['feature_set','model','accuracy','precision','recall','f1','roc_auc']]


,feature_set,model,accuracy,precision,recall,f1,roc_auc
0,full,baseline_majority,0.572931,0.572931,1.000000,0.728489,0.500000
1,full,logistic_regression,0.999932,0.999881,1.000000,0.999941,1.000000
2,full,decision_tree,1.000000,1.000000,1.000000,1.000000,1.000000
3,full,random_forest,1.000000,1.000000,1.000000,1.000000,1.000000
4,no_derived,baseline_majority,0.572931,0.572931,1.000000,0.728489,0.500000
5,no_derived,logistic_regression,0.999252,0.999051,0.999644,0.999348,0.999994
6,no_derived,decision_tree,0.998997,0.999466,0.998784,0.999125,0.999034
7,no_derived,random_forest,0.999864,0.999822,0.999941,0.999881,1.000000
8,url_only,baseline_majority,0.572931,0.572931,1.000000,0.728489,0.500000
9,url_only,logistic_regression,0.995887,0.993163,0.999703,0.996423,0.998135


## Handoff

Everything the report needs is now on disk:

- `results/cleaning_log.json` feeds the preprocessing half of Section 2
- `results/leakage_screen.csv` and `class_constancy.csv` feed the analytical argument
- `results/metrics_*.json` feed Section 3
- `figures/*.png` feed both

Report writers read these files. Do not retype numbers.